# Setup

In [10]:
#!pip install -r requirements.txt

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not ((repo_root / "Final").exists() and (repo_root / "Sprint 3").exists()):
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root.")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
from datetime import datetime
from typing import Any
import pandas as pd

from Final.config import default_config
from Final.paths import FINAL_ROOT
from Final.shared_utils import setup_logging, get_logger

from Final.models import (
    ExperimentState,
)
from Final.experiment_controller import ExperimentController
from Final.grid_search import GridSearchController
from Final.pipeline_runtime import execute_pipeline_section
from Final.artifact_store import (
    LocalArtifactStore,
    DriveRegistryArtifactStore,
    HybridArtifactStore,
)
from Final.gating import (
    evaluate_module_card,
    decide_module_status,
    module_cards_to_frame,
)
from Final.labeling.pipeline import LabelingPipeline, LabelingPipelineConfig, LabelingStorageConfig

/home/jovyan/.local/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/jovyan/.local/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/jovyan/.local/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on

SyntaxError: invalid syntax (pipeline.py, line 693)

In [ ]:
cfg = default_config()

logger = setup_logging(
    name="shrub",
    log_dir=cfg.output.logs_root,
    log_filename="main_pipeline.log",
    force=True,
)

logger.info("Initialized main pipeline notebook.")
logger.info("FINAL_ROOT = %s", FINAL_ROOT)

MAIN_OUTPUT_ROOT = cfg.output.root / "main"
MAIN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MAIN_ROOT = MAIN_OUTPUT_ROOT

MAIN_MANIFEST_DIR = MAIN_OUTPUT_ROOT / "manifests"
MAIN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS_ROOT = MAIN_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

In [13]:
controller = ExperimentController(
    experiment_root=cfg.output.root / "experiments",
    experiment_name="shrubwise_main",
)

state = controller.load_state()

pipelines = {}

local_store = LocalArtifactStore(
    repo_root=cfg.data.project_root,
    storage_root=cfg.output.root / "artifact_store_local",
)

USE_DRIVE = False

if USE_DRIVE:
    drive_store = DriveRegistryArtifactStore(
        repo_root=cfg.data.project_root,
        registry_path=cfg.output.root / "artifact_registry.yaml",
        drive_config_path=cfg.data.project_root / "drive_config.yaml",
        client_secrets_path=cfg.data.project_root / "client_secrets.json",
        credentials_path=cfg.data.project_root / "pydrive_credentials.json",
    )
    artifact_store = HybridArtifactStore(
        local_store=local_store,
        remote_store=drive_store,
    )
else:
    artifact_store = local_store

grid = GridSearchController(controller=controller)

## Labeling

In [14]:
labeling_cfg = LabelingPipelineConfig(
    sprint3_variant="revised",
    sprint3_variants=("original", "revised"),
    run_sprint3=True,
    max_ptx_per_site=1,
    force_rerun_sprint3=False,
    require_success_artifacts_sprint3=True,
    cleanup_ptx_after_all_variants=True,
    cleanup_stale_ptx_before_run=True,
    stale_ptx_days=2,
    use_shape_descriptors=True,
    use_temporal_confidence=True,
    use_boundary_confidence=True,
    boundary_confidence_mode="universal",
    use_transform_confidence=False,
    use_object_subspace_filter=False,
    rasterization_mode="circle",
    multires=cfg.raster.create_multires,
    site_reference_dates={},
    subspace_min_component_pixels=4,
    subspace_min_object_confidence=0.55,
    subspace_min_transform_confidence=0.50,
    subspace_min_temporal_confidence=0.40,
    subspace_max_height_m=3.5,
    force_rerun_sprint4=False,
    force_refresh_site_assets=False,
    nonfatal_qa_overlay=True,
    allow_adopt_global_outputs=False,
)

In [15]:
labeling_pipeline = LabelingPipeline(
    cfg,
    pipeline_config=labeling_cfg,
)

pipelines["labeling"] = labeling_pipeline

In [16]:
display(labeling_pipeline.pipeline_spec)
display(grid.pipeline_module_state_frame(labeling_pipeline))

PipelineSpec(pipeline_name='labeling', domain=<PipelineDomain.LABELING: 'label_engineering'>, stages=[StageSpec(name='sprint3', module_keys=['labeling.sprint3.execution'], cache_policy=CachePolicy(require_manifest=False, allow_legacy_reuse=False, retention_mode=<CacheRetentionMode.LEAN: 'lean'>, artifact_keys_to_prune=(), prune_after_success=False)), StageSpec(name='standardize', module_keys=['labeling.standardize.base'], cache_policy=CachePolicy(require_manifest=True, allow_legacy_reuse=False, retention_mode=<CacheRetentionMode.LEAN: 'lean'>, artifact_keys_to_prune=(), prune_after_success=False)), StageSpec(name='refine', module_keys=['labeling.refine.shape_descriptors', 'labeling.refine.temporal_confidence', 'labeling.refine.object_subspace_filter'], cache_policy=CachePolicy(require_manifest=True, allow_legacy_reuse=False, retention_mode=<CacheRetentionMode.LEAN: 'lean'>, artifact_keys_to_prune=(), prune_after_success=False)), StageSpec(name='transfer', module_keys=['labeling.transfe

,pipeline,stage_name,module_name,enabled,variant_name,params
0,labeling,sprint3,labeling.sprint3.execution,True,"('original', 'revised')","{'max_ptx_per_site': 1, 'force_rerun_sprint3':..."
1,labeling,standardize,labeling.standardize.base,True,default,{}
2,labeling,refine,labeling.refine.shape_descriptors,True,enabled,{}
3,labeling,refine,labeling.refine.temporal_confidence,True,enabled,{'site_reference_dates': {}}
4,labeling,refine,labeling.refine.object_subspace_filter,False,disabled,"{'subspace_min_object_confidence': 0.55, 'subs..."
5,labeling,transfer,labeling.transfer.base,True,default,{}
6,labeling,rasterize,labeling.rasterize.mode,True,circle,{}
7,labeling,rasterize,labeling.boundary_confidence,True,universal,{}
8,labeling,rasterize,labeling.mask_subspace_reduction,False,disabled,{'subspace_min_component_pixels': 4}
9,labeling,rasterize,labeling.multires_export,True,default,"{'multires': (1.0, 2.0, 5.0, 10.0)}"


In [17]:
labeling_space_df = grid.section_space_frame(labeling_pipeline).copy()
display(labeling_space_df)
print("Total labeling variants:", len(labeling_space_df))

,config_signature,sprint3_variants,use_temporal_confidence,boundary_confidence_mode,use_object_subspace_filter,max_ptx_per_site
0,074f2c2dc3a8,"original,revised",True,universal,True,1
1,0b9cdfd5f781,"original,revised",True,radial,False,1
2,16774e5c5bc7,"original,revised",True,radial,True,1
3,248d471f5527,revised,True,radial,False,1
4,38814738feeb,revised,True,universal,True,1
5,5725ea41a55c,"original,revised",False,radial,False,1
6,6930be0cde08,"original,revised",False,universal,True,1
7,88fb2c6936cb,"original,revised",True,universal,False,1
8,893d374e5995,"original,revised",False,universal,False,1
9,8d0d163a7065,revised,False,radial,True,1


Total labeling variants: 16


In [18]:
TRIAL_ID = "labeling_only_trial_001"

trial_path = controller.trial_path(TRIAL_ID)
if trial_path.exists():
    trial = controller.load_trial(TRIAL_ID)
else:
    trial = controller.create_trial(trial_id=TRIAL_ID)

trial

TrialRecord(trial_id='labeling_only_trial_001', created_at='2026-04-15T02:30:58+00:00', status='created', section_configs={'labeling': {'sprint3_variant': 'revised', 'use_shape_descriptors': True, 'use_temporal_confidence': True, 'use_boundary_confidence': True, 'use_transform_confidence': False, 'use_object_subspace_filter': False, 'rasterization_mode': 'circle', 'multires': [1.0, 2.0, 5.0, 10.0], 'force_rerun_sprint4': False, 'force_refresh_site_assets': False, 'nonfatal_qa_overlay': True, 'run_sprint3': True, 'sprint3_variants': ['original', 'revised'], 'max_ptx_per_site': 1, 'force_rerun_sprint3': False, 'require_success_artifacts_sprint3': True, 'cleanup_ptx_after_all_variants': True, 'cleanup_stale_ptx_before_run': True, 'stale_ptx_days': 2, 'boundary_confidence_mode': 'universal', 'site_reference_dates': {}, 'subspace_min_component_pixels': 4, 'subspace_min_object_confidence': 0.55, 'subspace_min_transform_confidence': 0.5, 'subspace_min_temporal_confidence': 0.4, 'subspace_max_

In [19]:
controller.set_section_config(
    trial,
    "labeling",
    labeling_pipeline.config_dict(),
)
controller.save_trial(trial)
trial

TrialRecord(trial_id='labeling_only_trial_001', created_at='2026-04-15T02:30:58+00:00', status='created', section_configs={'labeling': {'sprint3_variant': 'revised', 'use_shape_descriptors': True, 'use_temporal_confidence': True, 'use_boundary_confidence': True, 'use_transform_confidence': False, 'use_object_subspace_filter': False, 'rasterization_mode': 'circle', 'multires': (1.0, 2.0, 5.0, 10.0), 'force_rerun_sprint4': False, 'force_refresh_site_assets': False, 'nonfatal_qa_overlay': True, 'run_sprint3': True, 'sprint3_variants': ('original', 'revised'), 'max_ptx_per_site': 1, 'force_rerun_sprint3': False, 'require_success_artifacts_sprint3': True, 'cleanup_ptx_after_all_variants': True, 'cleanup_stale_ptx_before_run': True, 'stale_ptx_days': 2, 'boundary_confidence_mode': 'universal', 'site_reference_dates': {}, 'subspace_min_component_pixels': 4, 'subspace_min_object_confidence': 0.55, 'subspace_min_transform_confidence': 0.5, 'subspace_min_temporal_confidence': 0.4, 'subspace_max_

In [20]:
print("Experiment state:")
display(state)

print("Trial:")
display(trial)

print("Completed trials:")
display(grid.completed_trials_frame())

Experiment state:


ExperimentState(experiment_name='shrubwise_main', active_modules=[], notes='', raster_outputs={'labels': None, 'features': None, 'predictions': None, 'qa_overlays': None}, object_outputs={'objects': None, 'predicted_objects': None, 'source_provenance': None, 'quality_flags': None}, qa_outputs={}, section_status={})

Trial:


TrialRecord(trial_id='labeling_only_trial_001', created_at='2026-04-15T02:30:58+00:00', status='created', section_configs={'labeling': {'sprint3_variant': 'revised', 'use_shape_descriptors': True, 'use_temporal_confidence': True, 'use_boundary_confidence': True, 'use_transform_confidence': False, 'use_object_subspace_filter': False, 'rasterization_mode': 'circle', 'multires': (1.0, 2.0, 5.0, 10.0), 'force_rerun_sprint4': False, 'force_refresh_site_assets': False, 'nonfatal_qa_overlay': True, 'run_sprint3': True, 'sprint3_variants': ('original', 'revised'), 'max_ptx_per_site': 1, 'force_rerun_sprint3': False, 'require_success_artifacts_sprint3': True, 'cleanup_ptx_after_all_variants': True, 'cleanup_stale_ptx_before_run': True, 'stale_ptx_days': 2, 'boundary_confidence_mode': 'universal', 'site_reference_dates': {}, 'subspace_min_component_pixels': 4, 'subspace_min_object_confidence': 0.55, 'subspace_min_transform_confidence': 0.5, 'subspace_min_temporal_confidence': 0.4, 'subspace_max_

Completed trials:


,trial_id,status,n_section_runs
0,labeling_only_trial_001,created,0


In [22]:
labeling_result, state, runtime_stats = execute_pipeline_section(
    labeling_pipeline,
    state=state,
    artifact_store=artifact_store,
    push_remote=USE_DRIVE,
)

controller.record_section_result(
    trial,
    section_name="labeling",
    config_signature=labeling_pipeline.config_signature(),
    result=labeling_result,
)
controller.save_trial(trial)
controller.save_state(state)

labeling_result, runtime_stats

2026-04-15 08:53:17 | INFO     | shrub.labeling.sprint3_runner | Discovering remote PTX files for site=calaveras-big-trees at https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-calaveras-big-trees/original_TLS
2026-04-15 08:53:18 | INFO     | shrub.labeling.sprint3_runner | Discovering remote PTX files for site=dl-bliss at https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-dl-bliss/original_TLS
2026-04-15 08:53:18 | INFO     | shrub.labeling.sprint3_runner | Discovering remote PTX files for site=independence-lake at https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-independence-lake/original_TLS
2026-04-15 08:53:18 | INFO     | shrub.labeling.sprint3_runner | Discovering remote PTX files for site=pacific-union-college at https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-pacific-union-college/original_TLS
2026-04-15 08:53:19 | INFO     | shrub.labeling.sprint3_runner | Discovering remote PTX files for site=sedgwick at https://wifire-data.sdsc.edu/nc/public.


KeyboardInterrupt



In [ ]:
# stderr_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stderr.log"
# stdout_path = "/home/jovyan/work/Dry-shRub/shrub/Final/artifacts/labeling/calaveras-big-trees/sprint3/original/CATCU_0009_20250615_1/stdout.log"

# print("================ FULL STDERR ================")
# if os.path.exists(stderr_path):
#     with open(stderr_path, 'r') as f:
#         print(f.read())
# else:
#     print("stderr.log not found!")

# print("\n================ FULL STDOUT ================")
# if os.path.exists(stdout_path):
#     with open(stdout_path, 'r') as f:
#         print(f.read())
# else:
#     print("stdout.log not found!")

In [ ]:
print("Experiment state:")
display(state)

print("Trial:")
display(trial)

print("Completed trials:")
display(grid.completed_trials_frame())

## Features

In [ ]:
features_cfg = {
    "enabled": False,
    "feature_families": [],
    "notes": "Placeholder only for now.",
}

trial.features_config = features_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.features_config

In [ ]:
trial.features_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Features pipeline not implemented yet."],
}

state.section_status["features"] = "placeholder_not_run"
state.qa_outputs["features"] = trial.features_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.features_result

# Modeling

In [ ]:
modeling_cfg = {
    "enabled": False,
    "model_family": None,
    "notes": "Placeholder only for now.",
}

trial.modeling_config = modeling_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.modeling_config

In [ ]:
trial.modeling_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Modeling pipeline not implemented yet."],
}

state.section_status["modeling"] = "placeholder_not_run"
state.qa_outputs["modeling"] = trial.modeling_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.modeling_result

# Post-processing

In [ ]:
postprocessing_cfg = {
    "enabled": False,
    "steps": [],
    "notes": "Placeholder only for now.",
}

trial.postprocessing_config = postprocessing_cfg

save_trial(EXPERIMENT_NAME, trial)

trial.postprocessing_config

In [ ]:
trial.postprocessing_result = {
    "success": None,
    "status": "placeholder_not_run",
    "metrics": {},
    "qa_outputs": {"placeholder": True},
    "notes": ["Postprocessing pipeline not implemented yet."],
}

state.section_status["postprocessing"] = "placeholder_not_run"
state.qa_outputs["postprocessing"] = trial.postprocessing_result["qa_outputs"]

save_trial(EXPERIMENT_NAME, trial)
trial.postprocessing_result

# Finalize Trial

In [ ]:
trial.qa_summary = {
    "integrity": {
        "labeling": trial.labeling_result.get("status"),
        "features": trial.features_result.get("status"),
        "modeling": trial.modeling_result.get("status"),
        "postprocessing": trial.postprocessing_result.get("status"),
    },
    "section_level": {
        "labeling": "placeholder",
        "features": "placeholder",
        "modeling": "placeholder",
        "postprocessing": "placeholder",
    },
    "cross_section": {
        "label_to_model_feedback": "placeholder",
        "feature_to_model_feedback": "placeholder",
        "end_to_end_feedback": "placeholder",
    },
}

save_trial(EXPERIMENT_NAME, trial)
trial.qa_summary

In [ ]:
labeling_object_rows = trial.labeling_result.get("metrics", {}).get("n_object_rows", 0)
labeling_artifact_rows = trial.labeling_result.get("metrics", {}).get("n_artifact_rows", 0)

trial.score_summary = {
    "labeling_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
    "features_score_placeholder": None,
    "modeling_score_placeholder": None,
    "postprocessing_score_placeholder": None,
    "composite_score_placeholder": float(labeling_object_rows > 0) + float(labeling_artifact_rows > 0),
}

save_trial(EXPERIMENT_NAME, trial)
trial.score_summary

In [ ]:
trial.status = "partial" if (
    trial.features_result.get("status") == "placeholder_not_run"
    or trial.modeling_result.get("status") == "placeholder_not_run"
    or trial.postprocessing_result.get("status") == "placeholder_not_run"
) else "completed"

save_trial(EXPERIMENT_NAME, trial)

trial

In [ ]:
trial_record_min = {
    "trial_id": trial.trial_id,
    "created_at": trial.created_at,
    "status": trial.status,
    "labeling_config": trial.labeling_config,
    "features_config": trial.features_config,
    "modeling_config": trial.modeling_config,
    "postprocessing_config": trial.postprocessing_config,
    "score_summary": trial.score_summary,
}

registry.trials = [t for t in registry.trials if t["trial_id"] != trial.trial_id]
registry.trials.append(trial_record_min)

score = trial.score_summary.get("composite_score_placeholder")
if score is not None:
    if registry.best_score is None or score > registry.best_score:
        registry.best_score = score
        registry.best_trial_id = trial.trial_id

save_registry(registry)

registry

In [ ]:
trials_df = pd.DataFrame(registry.trials)

if not trials_df.empty:
    if "score_summary" in trials_df.columns:
        trials_df["composite_score_placeholder"] = trials_df["score_summary"].apply(
            lambda x: x.get("composite_score_placeholder") if isinstance(x, dict) else None
        )

    display(
        trials_df[
            ["trial_id", "created_at", "status", "composite_score_placeholder"]
        ].sort_values("trial_id")
    )

    print("Best trial:", registry.best_trial_id)
    print("Best score:", registry.best_score)
else:
    print("No trials recorded yet.")

In [ ]:
INSPECT_TRIAL_ID = trial.trial_id  # change manually

inspect_trial = load_trial(EXPERIMENT_NAME, INSPECT_TRIAL_ID)
inspect_trial